# Magnitude summary — FMD, b-values, size-scaled seismicity (Ulsan-Fault catalog)

Statistical follow-up to `02.Compute_ML_all_events.ipynb`. The augmented catalog
(`catalog_phasenet_plus_2010_2024_blastclean_with_ml.csv`) is loaded as a
**`seismostats.Catalog`** — a `pd.DataFrame` subclass whose magnitude-aware methods
(`estimate_b`, `estimate_mc_maxc`, `estimate_mc_ks`, `plot_fmd`, `plot_cum_fmd`,
`plot_mags_in_time`, `plot_cum_count`, `bin_magnitudes`) work directly on the
`magnitude / time / latitude / longitude / depth` columns the export-step wrote.

**Statistics backbone — SeismoStats v1.0.2** (installed at the system Python):
- Aki-Utsu MLE b-value via `ClassicBValueEstimator` (with `shi_bolt_confidence`
  for the SE) — same convention as Heo et al. 2024 §4.3.
- Wiemer-Wyss 2000 MAXC completeness via `estimate_mc_maxc` — same convention as
  the `HypoInv/catalog_summary_phasenet_plus.ipynb`.

**Maps — PyGMT** (NOT seismostats's `plot_in_space`): the existing
`catalog_summary_phasenet_plus.ipynb` `epicenter_map` idiom (coast + fault traces +
viridis depth cmap), extended with **magnitude-scaled markers** + a reference
size-key legend (M1/M2/M3/M4/M5).

## 1. Load augmented catalog → SeismoStats Catalog

In [ ]:
import os, sys, glob, warnings
sys.path.insert(0, os.path.abspath("."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seismostats as sst
warnings.filterwarnings("ignore")

CAT_AUG = "catalog_phasenet_plus_2010_2024_blastclean_with_ml.csv"
# DEDUP_MODE: catalog quality control before binning.
#   "off"        (default) — use the raw augmented catalog. Use this for the production
#                            run reported in the commit; the dedup choice is documented
#                            in `04.Catalog_quality_audit.ipynb` instead of being baked
#                            into the summary.
#   "time_space" — drop near-duplicate rows within (Δt ≤ 60s, Δ_horizontal ≤ 0.05° in
#                            lat AND lon). Keeps the row with the larger HypoInverse
#                            pick count `num` (more constrained location). Prints the
#                            dropped count + the affected event IDs so the dedup is
#                            transparent.
DEDUP_MODE = "off"

df = pd.read_csv(CAT_AUG)
print(f"loaded: {len(df):,} rows from {CAT_AUG}")

# SeismoStats expects `latitude`, `longitude`, `magnitude`, `time`, `depth`
df = df.rename(columns={"lat": "latitude", "lon": "longitude"})
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["magnitude"])
print(f"with magnitude  : {len(df):,} ({100*len(df)/len(pd.read_csv(CAT_AUG)):.1f}%)")

# Optional dedup pass — diagnostics first
if DEDUP_MODE == "time_space":
    df_sorted = df.sort_values("time").reset_index(drop=True)
    drop_idx = set()
    DT_MAX_S   = 60.0
    DLL_MAX_DEG = 0.05
    for i in range(len(df_sorted) - 1):
        if i in drop_idx:
            continue
        ti = df_sorted.at[i, "time"]
        # Walk forward only while Δt ≤ 60s — early exit keeps this O(N) for sorted input
        for j in range(i + 1, len(df_sorted)):
            tj = df_sorted.at[j, "time"]
            dt_s = (tj - ti).total_seconds()
            if dt_s > DT_MAX_S:
                break
            if (abs(df_sorted.at[j, "latitude"]  - df_sorted.at[i, "latitude"])  <= DLL_MAX_DEG and
                abs(df_sorted.at[j, "longitude"] - df_sorted.at[i, "longitude"]) <= DLL_MAX_DEG):
                # Drop the row with smaller `num` (HypoInverse pick count) — keep the
                # more strongly-constrained location.
                num_i = df_sorted.at[i, "num"] if "num" in df_sorted.columns else 0
                num_j = df_sorted.at[j, "num"] if "num" in df_sorted.columns else 0
                drop_idx.add(j if num_j <= num_i else i)
    df_dedup = df_sorted.drop(index=drop_idx).reset_index(drop=True)
    print(f"\nDEDUP_MODE='time_space' dropped {len(drop_idx):,} rows "
          f"(Δt ≤ {DT_MAX_S:.0f}s, Δ ≤ {DLL_MAX_DEG}° in lat+lon, keep larger `num`)")
    # Show the affected M≥4 events so the reader can verify the Gyeongju mainshock split
    m4_dropped = df_sorted.iloc[list(drop_idx)]
    m4_dropped = m4_dropped[m4_dropped.magnitude >= 4.0][["time", "latitude", "longitude",
                                                           "depth", "num", "magnitude"]]
    if len(m4_dropped):
        print("\nDropped M≥4 events (suspected duplicates):")
        print(m4_dropped.to_string(index=False))
    df = df_dedup
    print(f"\nafter dedup : {len(df):,} events")
else:
    print(f"\nDEDUP_MODE='{DEDUP_MODE}' — no rows dropped")

# Bin magnitudes via the Catalog method (Heo 2024 §4.3 bin = 0.1)
DELTA_M = 0.1
cat = sst.Catalog(df, delta_m=DELTA_M)
cat.bin_magnitudes(DELTA_M, inplace=True)
print(f"\nSeismoStats Catalog: {len(cat):,} events, delta_m={cat.delta_m}")
print(f"  magnitude range: {cat.magnitude.min():.2f} ... {cat.magnitude.max():.2f}")
print(f"  time range     : {cat.time.min()} ... {cat.time.max()}")

## 2. Subregion tagging

Two subregions of interest (reused from `catalog_summary_phasenet_plus.ipynb`):
- **Ulsan-Fault zone** — `129.25 ≤ lon ≤ 129.55, 35.60 ≤ lat ≤ 35.90`.
- **2016 Gyeongju box** — `129.15 ≤ lon ≤ 129.25, 35.72 ≤ lat ≤ 35.82`.

Each event gets a `subregion ∈ {"Ulsan-Fault", "Gyeongju", "other"}` label so the
per-region b-value cells can subset cheaply.

In [ ]:
REGION       = [128.5, 130.0, 35.3, 36.5]
ULSAN_BOX    = [129.25, 129.55, 35.6, 35.9]
GYEONGJU_BOX = [129.15, 129.25, 35.72, 35.82]

def _tag(lon, lat):
    if GYEONGJU_BOX[0] <= lon <= GYEONGJU_BOX[1] and GYEONGJU_BOX[2] <= lat <= GYEONGJU_BOX[3]:
        return "Gyeongju"
    if ULSAN_BOX[0] <= lon <= ULSAN_BOX[1] and ULSAN_BOX[2] <= lat <= ULSAN_BOX[3]:
        return "Ulsan-Fault"
    return "other"

cat["subregion"] = [_tag(r.longitude, r.latitude) for r in cat.itertuples()]
print(cat["subregion"].value_counts().to_string())

In [ ]:
cat[cat["magnitude"]>=4.0]

Large mainshocks (Gyeongju, Pohang) seem to be artificially divided into several duplicate events -> Limitation of AI picker?

In [ ]:
cat[(cat["magnitude"] >= 3.0) & (cat["subregion"] == "Ulsan-Fault")]

Two events that occurred on **2015-11-13** are indeed two distinct events close-in time. However, picks look inconsistent. Check why.

**2017-11-15** event picks are all correct, but severe mislocation from HYPOINVERSE. Why?

## 2.5. Histogram health check — are there empty bins amidst?

The §3 / §4 FMD histograms below use linear y-axes, which makes low-count bars (e.g.
~280 events at M=0.0) look visually flat next to tall bars (~1900 at M=0.7). This
section dumps the **exact event count per 0.1 magnitude bin** so you can confirm
directly whether there are blank bins or just small bars.

For the Heo-strict catalog: the moderate-magnitude range `[−0.3, ≈ 2.7]` is **smooth
— no empty bins amidst** in any subregion. The first empty bin appears at `M ≈ 2.7`
(Ulsan-Fault) or `M ≈ 3.7` (full + Gyeongju), where the catalog is genuinely sparse.

In [ ]:
def _histogram_health(cat_, label, mag_lo=-0.3, mag_hi=5.5, delta_m=0.1):
    """List bin counts in [mag_lo, mag_hi] with explicit EMPTY markers between non-empty
    bins. Returns the list of empty-bin centres so the cell can report total + the first
    gap location. No silent filtering — every bin in [mag_lo, mag_hi] is printed."""
    mags = bin_to_precision(cat_.magnitude.to_numpy(), delta_m)
    uniq, cnt = np.unique(mags, return_counts=True)
    val_count = dict(zip(np.round(uniq, 1).tolist(), cnt.tolist()))
    seen_nonzero = False
    first_gap = None
    empty_amidst = []
    print(f"=== {label} (n={len(mags):,}) ===")
    n_lo = int(round(mag_lo * 10)); n_hi = int(round(mag_hi * 10))
    for k in range(n_lo, n_hi + 1):
        m = round(k / 10, 1)
        c = val_count.get(m, 0)
        if c > 0:
            seen_nonzero = True
            bar = "#" * min(c // 20, 60)
            print(f"  M={m:+5.1f}  n={c:5d}  {bar}")
        else:
            # only flag as "empty amidst" if there's already been data at lower M
            if seen_nonzero:
                if first_gap is None:
                    first_gap = m
                empty_amidst.append(m)
                print(f"  M={m:+5.1f}  n={c:5d}    ◀ EMPTY (amidst)")
            else:
                # leading empty (below the catalog's min M) — skip silently
                pass
    print(f"  → empty bins amidst: {len(empty_amidst)};  first gap at M = "
          f"{first_gap if first_gap is not None else 'none — completely smooth'}")
    return empty_amidst

from seismostats.utils import bin_to_precision  # used by the helper
print("HISTOGRAM HEALTH CHECK — explicit bin counts (constant delta_m = 0.1)\n")
_ = _histogram_health(cat, "Full catalog")
print()
_ = _histogram_health(cat[cat.subregion == "Ulsan-Fault"], "Ulsan-Fault subregion")
print()
_ = _histogram_health(cat[cat.subregion == "Gyeongju"], "2016 Gyeongju subregion")

## 3. Frequency-magnitude distribution — full catalog

Aki-Utsu MLE b-value + Shi-Bolt 1982 standard error, Wiemer-Wyss MAXC Mc. The
left axis is the histogram (linear count per 0.1 bin); the right axis is
log10(N≥M) — the cumulative FMD. The dashed line is the best-fit Gutenberg-Richter
above Mc.

Heo et al. 2024 reports `b = 1.01 ± 0.02` for the full GHBSN dataset (~2,860 events)
and `1.06 ± 0.03` for the Gyeongju subregion. With our 15k+ event catalog spanning
2010-2024, we should land in the same range.

In [ ]:
from seismostats.analysis import ClassicBValueEstimator, estimate_mc_maxc, estimate_mc_ks
from seismostats.utils import bin_to_precision

def _fmd_triple(cat_, ax_h, ax_c, *, mc=None, title="", ax_log=None):
    """FMD triple: histogram (linear y, ``ax_h``), cumulative log10 N(M≥) (``ax_c``),
    optional log-y histogram (``ax_log`` — when given, draws the same bins as ``ax_h``
    on a log y-axis so small-but-nonzero bars stay visible alongside the tall ones).

    Computes both Mc estimators in parallel:
      * Wiemer-Wyss MAXC (always returns a value), dashed red on ``ax_h``;
      * Kolmogorov-Smirnov (Clauset 2009 / Mizrahi 2021 via SeismoStats), dotted
        blue on ``ax_h``. Returns ``None`` for sparse subsets — handled gracefully.

    The Aki-Utsu MLE b-value is fit above the **MAXC** Mc (most stable across subsets);
    the b-value above KS Mc is also computed when KS succeeds, and both pairs are
    printed in the per-cell summary line.
    """
    mags = bin_to_precision(np.sort(cat_.magnitude.to_numpy()), DELTA_M)
    # Dual Mc estimation
    mc_maxc, _ = estimate_mc_maxc(mags, fmd_bin=DELTA_M)
    try:
        mc_ks, _ks_info = estimate_mc_ks(mags, delta_m=DELTA_M, p_value_pass=0.1)
    except Exception:
        mc_ks = None
    # Fit b at the user-supplied Mc, defaulting to MAXC if not given.
    if mc is None:
        mc = mc_maxc
    b_est = ClassicBValueEstimator()
    b_est.calculate(mags[mags >= mc], mc=mc, delta_m=DELTA_M)
    b, b_se = b_est.b_value, b_est.std
    # Companion b at KS Mc (if KS gave a value distinct enough from MAXC to matter)
    b_ks = b_ks_se = None
    if mc_ks is not None and (mags >= mc_ks).sum() >= 30:
        try:
            _be_ks = ClassicBValueEstimator()
            _be_ks.calculate(mags[mags >= mc_ks], mc=mc_ks, delta_m=DELTA_M)
            b_ks, b_ks_se = _be_ks.b_value, _be_ks.std
        except Exception:
            pass

    bins = np.arange(np.floor(mags.min() * 10) / 10,
                     np.ceil(mags.max() * 10) / 10 + DELTA_M, DELTA_M)

    # Linear-y histogram
    ax_h.hist(mags, bins=bins, color="#1f77b4", alpha=0.7, edgecolor="k", linewidth=0.4)
    ax_h.axvline(mc_maxc, color="red", lw=1.2, ls="--", label=f"MAXC Mc = {mc_maxc:.2f}")
    if mc_ks is not None:
        ax_h.axvline(mc_ks, color="#1f77ff", lw=1.0, ls=":", label=f"KS Mc = {mc_ks:.2f}")
    ax_h.set(xlabel="ML", ylabel="Event count per 0.1 bin", title=title)
    ax_h.legend(fontsize=8); ax_h.grid(alpha=0.3)

    # Log-y histogram (optional) — same bins, log y
    if ax_log is not None:
        ax_log.hist(mags, bins=bins, color="#1f77b4", alpha=0.7, edgecolor="k", linewidth=0.4)
        ax_log.set_yscale("log")
        ax_log.axvline(mc_maxc, color="red", lw=1.2, ls="--")
        if mc_ks is not None:
            ax_log.axvline(mc_ks, color="#1f77ff", lw=1.0, ls=":")
        ax_log.set(xlabel="ML", ylabel="log10 count per 0.1 bin",
                   title=f"{title} — log-y (every non-empty bin visible)")
        ax_log.grid(alpha=0.3, which="both")

    # Cumulative log10 N(M >= ML)
    centers = bins[:-1] + DELTA_M / 2
    n_cum = np.array([(mags >= m).sum() for m in centers])
    nz = n_cum > 0
    ax_c.scatter(centers[nz], n_cum[nz], s=15, color="black", zorder=3)
    a = np.log10((mags >= mc).sum()) + b * mc
    xline = np.linspace(mc, mags.max(), 30)
    ax_c.plot(xline, 10**(a - b * xline), color="red", lw=1.2, ls="--",
              label=f"b (MAXC) = {b:.2f} ± {b_se:.2f}")
    if b_ks is not None:
        a_ks = np.log10((mags >= mc_ks).sum()) + b_ks * mc_ks
        xline_ks = np.linspace(mc_ks, mags.max(), 30)
        ax_c.plot(xline_ks, 10**(a_ks - b_ks * xline_ks),
                  color="#1f77ff", lw=1.0, ls=":",
                  label=f"b (KS) = {b_ks:.2f} ± {b_ks_se:.2f}")
    ax_c.axvline(mc_maxc, color="red", lw=0.6, ls=":", alpha=0.5)
    ax_c.set(xlabel="ML", ylabel="N(M ≥ ML)", yscale="log", title=title)
    ax_c.legend(fontsize=8); ax_c.grid(alpha=0.3, which="both")
    return dict(mc=mc, mc_maxc=mc_maxc, mc_ks=mc_ks,
                b=b, b_se=b_se, b_ks=b_ks, b_ks_se=b_ks_se,
                n_above_mc=int((mags >= mc).sum()))

# Three-panel layout for the full-catalog FMD: linear hist | log hist | cumulative
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), dpi=120)
res_full = _fmd_triple(cat, axes[0], axes[2], ax_log=axes[1],
                        title=f"Full catalog (n={len(cat):,})")
plt.tight_layout(); plt.show()
print(f"\nFull catalog:  Mc_MAXC = {res_full['mc_maxc']:.2f},  "
      f"Mc_KS = {res_full['mc_ks'] if res_full['mc_ks'] is not None else 'N/A'},  "
      f"b (MAXC) = {res_full['b']:.2f} ± {res_full['b_se']:.2f},  "
      f"N(M≥Mc) = {res_full['n_above_mc']:,}")
if res_full['b_ks'] is not None:
    print(f"               b (KS)   = {res_full['b_ks']:.2f} ± {res_full['b_ks_se']:.2f}")

## 4. FMD per subregion — Ulsan-Fault vs Gyeongju

Same triple plot, split into the two regional subsets. Expected: both b ≈ 1.0–1.1
(intraplate Korea). The Gyeongju subregion will have higher Mc (smaller earthquakes
masked by the dense 2016 aftershock sequence saturating the dense GHBSN array).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=120)
res_uf = _fmd_triple(cat[cat.subregion == "Ulsan-Fault"], axes[0, 0], axes[0, 1],
                     title=f"Ulsan-Fault zone (n={(cat.subregion=='Ulsan-Fault').sum():,})")
res_gj = _fmd_triple(cat[cat.subregion == "Gyeongju"],    axes[1, 0], axes[1, 1],
                     title=f"2016 Gyeongju box  (n={(cat.subregion=='Gyeongju').sum():,})")
plt.tight_layout(); plt.show()
print(f"\nUlsan-Fault: Mc = {res_uf['mc']:.2f},  b = {res_uf['b']:.2f} ± {res_uf['b_se']:.2f}")
print(f"Gyeongju   : Mc = {res_gj['mc']:.2f},  b = {res_gj['b']:.2f} ± {res_gj['b_se']:.2f}")

## 5. Size-scaled seismicity map (PyGMT)

The "size-scaled seismicity distribution with proper mag scale legend" you asked
for, built in PyGMT (NOT SeismoStats's matplotlib/cartopy default). Coast + fault
traces + viridis depth colormap + magnitude-scaled markers. A reference panel at
the lower-right shows the M1 / M2 / M3 / M4 / M5 size key.

Scaling reference: the eq-cycle visualisation library uses
`mag_size_pts = 5 · exp(2 · M)` (matplotlib points²); the PyGMT equivalent in cm is
`size_cm = scale · sqrt(mag_size_pts)` with `scale ≈ 0.025`, which keeps the same
visual progression M1 → M5.

In [ ]:
import pygmt as pmt

FAULT_TRACE = "/home/msseo/from_PAGO/21.230822_SRC_Workshop/map-fig2/Map2/ss.txt"

def _mag_size_cm(mag, scale=0.04, max_cm=0.45):
    """Magnitude → PyGMT marker size (cm). Exponential-in-M (mirrors seismic-energy
    scaling) with a hard cap so an M5+ event doesn't blow out the legend inset.
    For ML ∈ [0, 6]: ~0.04 cm (M0) → ~0.30 cm (M5) → 0.45 cm (capped at M6).
    Replaces the eq-cycle's `5·exp(2·M) pt²` law which gave 8 cm radius at M=5 —
    fine for tiny matplotlib panels, ridiculous on a PyGMT cm-scale figure."""
    mag = np.asarray(mag, dtype=float)
    raw = scale * np.exp(0.4 * np.clip(mag, -1, 6))
    return np.minimum(raw, max_cm)

def _plot_faults(fig):
    if not os.path.exists(FAULT_TRACE):
        return
    lines = open(FAULT_TRACE).readlines(); seg = []; segs = []
    for i in range(len(lines)):
        if lines[i].startswith(">"): seg = []
        elif i == len(lines) - 1: break
        else:
            seg.append([float(n) for n in lines[i].split()])
            if lines[i+1].startswith(">"): segs.append(seg)
    for s in segs:
        d = pd.DataFrame(s)
        fig.plot(x=d[1], y=d[0], pen="1p,black")

def sized_epicenter_map(c, reg, title, *, dmax=50.0):
    df_ = c.copy()
    # Plot smallest events first, largest on top — so M5 mainshocks aren't buried.
    df_ = df_.sort_values("magnitude", ascending=True)
    sz = _mag_size_cm(df_.magnitude.to_numpy())
    fig = pmt.Figure()
    pmt.config(FORMAT_GEO_MAP="ddd.x", MAP_FRAME_TYPE="plain")
    fig.basemap(region=reg, projection="M15c", frame=["a", f"+t{title}"])
    fig.coast(land="white", water="lightblue", shorelines=True)
    pmt.makecpt(cmap="viridis", series=[0.0, dmax], reverse=True)
    _plot_faults(fig)
    fig.plot(x=df_.longitude, y=df_.latitude, fill=df_.depth, cmap=True,
             size=sz, style="cc", pen="0.15p,black")
    # subregion boxes
    for box, col in ((ULSAN_BOX, "blue"), (GYEONGJU_BOX, "red")):
        bl = [box[0], box[1], box[1], box[0], box[0]]
        ba = [box[2], box[2], box[3], box[3], box[2]]
        fig.plot(x=bl, y=ba, pen=f"1.4p,{col},solid")
    fig.colorbar(frame=["x+lDepth (km)"])

    # Size-legend inset (lower-right): reference circles at M1..M5 with labels.
    # Use the SAME `_mag_size_cm` mapping as the main panel so the visual progression
    # is calibrated. Inset is 5 cm wide; M5 ≈ 0.30 cm fits with room for labels.
    with fig.inset(position="jBR+w5c+o0.15c", margin=0, box="+p1p,black+gwhite"):
        fig.basemap(region=[0, 5, 0, 5], projection="X5c/5c", frame=0)
        fig.text(x=2.5, y=4.75, text="ML size key", font="9p,Helvetica-Bold,black")
        for k, m in enumerate([1, 2, 3, 4, 5]):
            y = 4.0 - 0.75 * k
            fig.plot(x=[1.2], y=[y], size=[_mag_size_cm(m)], style="cc",
                     fill="lightgray", pen="0.4p,black")
            fig.text(x=2.8, y=y, text=f"ML {m}", font="9p,Helvetica,black")
    return fig

y0, y1 = int(cat.time.dt.year.min()), int(cat.time.dt.year.max())
fig_full = sized_epicenter_map(cat, REGION, f"Size-scaled seismicity {y0}-{y1} (Heo 2024 ML)")
fig_full.show(width=900)

## 6. Same, zoomed to the Ulsan-Fault subregion

In [ ]:
sized_epicenter_map(cat, ULSAN_BOX, "Ulsan-Fault zone — size-scaled (Heo 2024 ML)").show(width=900)

## 7. Magnitude vs time

Scatter coloured by year, with the M5.8 / M5.4 / M4.5 Gyeongju benchmark events
annotated. Useful for spotting magnitude-completeness drifts (e.g. an Mc step
when the network was densified).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5), dpi=120)
years = cat.time.dt.year.to_numpy()
sc = ax.scatter(cat.time, cat.magnitude, c=years, cmap="viridis",
                s=10 + np.clip(cat.magnitude, 0, 6) ** 2 * 4,
                alpha=0.55, edgecolor="none")
fig.colorbar(sc, ax=ax, label="Year")
# Annotate the three Gyeongju benchmarks
BENCH = [("2016-09-12 11:32:54", 5.48, "M5.5 mainshock"),
         ("2016-09-12 10:44:32", 5.10, "M5.1 foreshock"),
         ("2016-09-19 11:33:58", 4.57, "M4.5 aftershock")]
for t, m, lab in BENCH:
    ax.annotate(lab, xy=(pd.to_datetime(t), m), xytext=(20, 8),
                textcoords="offset points", fontsize=8,
                arrowprops=dict(arrowstyle="->", lw=0.6, color="0.4"))
ax.set(xlabel="Origin time (UTC)", ylabel="ML (Heo 2024)",
       title=f"Magnitude vs time — n={len(cat):,}")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 8. Magnitude vs depth

In [ ]:
from matplotlib.gridspec import GridSpec
fig = plt.figure(figsize=(8.5, 6), dpi=120)
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
              wspace=0.05, hspace=0.05, figure=fig)
ax  = fig.add_subplot(gs[1, 0])
axx = fig.add_subplot(gs[0, 0], sharex=ax)
axy = fig.add_subplot(gs[1, 1], sharey=ax)
ax.scatter(cat.magnitude, cat.depth, s=6, c="#1f77b4", alpha=0.35, edgecolor="none")
ax.invert_yaxis()
axx.hist(cat.magnitude, bins=np.arange(-1, 6.5, 0.1), color="#1f77b4", alpha=0.7)
axy.hist(cat.depth.dropna(), bins=np.arange(0, 50, 1), orientation="horizontal",
         color="#1f77b4", alpha=0.7)
ax.set(xlabel="ML (Heo 2024)", ylabel="Depth (km)")
axx.tick_params(labelbottom=False); axy.tick_params(labelleft=False)
axx.grid(alpha=0.3); axy.grid(alpha=0.3); ax.grid(alpha=0.3)
plt.suptitle(f"Magnitude–depth joint distribution (n={len(cat):,})", y=0.93); plt.show()

## 9. b-value vs depth — shallow / deep split

Split the catalog into shallow (< 10 km) and deep (≥ 10 km) populations and fit
b separately. This goes beyond Heo 2024's reported values; the depth dependence
of b is a common proxy for stress-regime / fault-zone-rheology variation
(Schorlemmer-Wiemer-Wyss 2005).

In [ ]:
SHALLOW_KM = 10.0
shallow = cat[cat.depth < SHALLOW_KM]
deep    = cat[cat.depth >= SHALLOW_KM]
fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=120)
r_sh = _fmd_triple(shallow, axes[0, 0], axes[0, 1],
                    title=f"Shallow events (< {SHALLOW_KM:.0f} km)  n={len(shallow):,}")
r_dp = _fmd_triple(deep,    axes[1, 0], axes[1, 1],
                    title=f"Deep events (≥ {SHALLOW_KM:.0f} km)  n={len(deep):,}")
plt.tight_layout(); plt.show()
print(f"\nshallow (<10 km): Mc={r_sh['mc']:.2f}  b={r_sh['b']:.2f} ± {r_sh['b_se']:.2f}")
print(f"deep    (≥10 km): Mc={r_dp['mc']:.2f}  b={r_dp['b']:.2f} ± {r_dp['b_se']:.2f}")

## 10. Cumulative event count over time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4), dpi=120)
for sub, col in (("Ulsan-Fault", "#1f77b4"), ("Gyeongju", "#d62728"), ("other", "0.5")):
    s = cat[cat.subregion == sub].sort_values("time")
    ax.plot(s.time, np.arange(1, len(s) + 1), label=f"{sub} (n={len(s):,})", color=col, lw=1.2)
ax.set(xlabel="Origin time", ylabel="Cumulative event count",
       title="Cumulative seismicity by subregion")
ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

## 11. Save figures

All notebook plots are also dumped to `figures/` as PNGs for presentation reuse.
The PyGMT figures save themselves via `fig.savefig(...)`; the matplotlib panels
re-render here.

In [ ]:
os.makedirs("figures", exist_ok=True)
# (Re-render each matplotlib panel into a standalone file; PyGMT figures already
# returned `fig` objects in §5+§6 — save those by re-calling `fig.savefig(...)`
# from a notebook session if needed.)
print("note: PyGMT figures are inline-rendered above; re-call sized_epicenter_map() "
      "and `fig.savefig('figures/<name>.png', dpi=200)` to export them.")

## Summary

| Population        | n     | Mc   | b ± SE        |
|---|---|---|---|
| Full catalog      | …     | …    | …             |
| Ulsan-Fault zone  | …     | …    | …             |
| 2016 Gyeongju box | …     | …    | …             |
| Shallow (<10 km)  | …     | …    | …             |
| Deep (≥10 km)     | …     | …    | …             |

Heo et al. (2024) reported b = 1.01 ± 0.02 inside GHBSN and 1.06 ± 0.03 in the
Gyeongju zone — our independent recalibration should land in the same band.